In [ ]:

import pandas as pd
#%pip install great_expectations
#%pip install sqlalchemy-bigquery google-cloud-bigquery
import great_expectations as gx
from great_expectations import expectations as gxe
 

# 1. Initialize Context
context = gx.get_context()
  

# 2. Setup BigQuery Connection
GCP_PROJECT_ID = "ds6-module-2-olist-analytics"
DATASET_NAME = "olist_dbt_dev"
GCS_BUCKET = "plexiform-resale-gx-datadocs"
 
connection_string = f"bigquery://{GCP_PROJECT_ID}/{DATASET_NAME}"

datasource = context.data_sources.add_sql(
    name="bigquery_datasource",
    connection_string=connection_string,
)

# 3. Define Table Quality Rules Matrix
TABLE_CONFIGS = [
    {
        "table_name": "fct_orders",
        "asset_name": "fct_orders",
        "not_null_cols": ["order_id", "customer_id", "order_status", "primary_payment_type"],
        "unique_cols": ["order_id"],
    },
    {
        "table_name": "dim_customers",
        "asset_name": "dim_customers",
        "not_null_cols": ["customer_id","customer_unique_id"],
        "unique_cols": ["customer_id"],
    },
    {
        "table_name": "fct_order_items",
        "asset_name": "fct_order_items",
        "not_null_cols": ["order_id", "product_id", "order_item_id", "seller_id"],
        "unique_cols": ["order_item_id"],
        "numeric_range_cols": [
            {"column": "price", "min": 0, "max": 100000}],
    },
     {
        "table_name": "category_performance_monthly",
        "asset_name": "category_performance_monthly",
        "not_null_cols": ["category_name_english", "order_month", "revenue"],
        "unique_cols": ["category_name_english", "order_month"]
        "ignore_row_if": ["any_value_is_missing"],
       
        "date_range_cols": [
            {"column": "order_month", "min": "2000-01-01", "max": "2026-12-31"}
        ],
        "numeric_range_cols": [
            {
                "column": "order_count",
                "min_value": 1,
                 "max_value": 1000000
            },
            {
                "column": "avg_freight_price_ratio",
                "min_value": 0,
                "strict_min": False
            }
        ],
        },
   
        
]


validation_definitions = []

# 4. Dynamic Loop Across Tables
for config in TABLE_CONFIGS:
    table_name = config["table_name"]
    asset_name = config["asset_name"]

    # Add Table Asset
    asset = datasource.add_table_asset(
        name=asset_name, table_name=table_name, schema_name=DATASET_NAME
    )
    batch_def = asset.add_batch_definition_whole_table(
        f"{asset_name}_batch_def"
    )

    # Add or Get Expectation Suite per table
    suite_name = f"{asset_name}_suite"
    try:
        suite = context.suites.get(suite_name)
    except Exception:
        suite = context.suites.add(gx.ExpectationSuite(name=suite_name))

    # Dynamically attach expectations based on configuration
    for col in config["not_null_cols"]:
        suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col))

    for col in config["unique_cols"]:
        suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column=col))
               
   # for col in config["numeric_range_cols"]:
    #    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column=col, min_value=col["min"], max_value=col["max"]))

    #numerical range checks
    for rule in config.get("numeric_range_cols", []):
        suite.add_expectation(
            gx.expectations.ExpectColumnValuesToBeBetween(
                column=rule["column"],
                min_value=rule.get("min_value", rule.get("min")),
                max_value=rule.get("max_value", rule.get("max")),   
            )
        )


# 3. DATE RANGE (Safe against missing key)
    for rule in config.get("date_range_cols", []):
        suite.add_expectation(
            gx.expectations.ExpectColumnValuesToBeBetween(
                column=rule["column"],
                min_value=rule.get("min"),
                max_value=rule.get("max"),
            )
        )


    # Create Validation Definition for this table
    val_def = context.validation_definitions.add_or_update(
        gx.ValidationDefinition(
            name=f"{asset_name}_validation",
            data=batch_def,
            suite=suite,
        )
    )
    validation_definitions.append(val_def)

# 5. Single Master Checkpoint to run all table validations at once
checkpoint = context.checkpoints.add_or_update(
    gx.Checkpoint(
        name="multi_table_pipeline_checkpoint",
        validation_definitions=validation_definitions,
    )
)

# 6. Execute Checkpoint
results = checkpoint.run()

# Build Data Docs without triggering browser auto-open
context.build_data_docs()

# Fetch the exact file path to the generated index.html
docs_sites = context.get_docs_sites_urls()
for site in docs_sites:
    print(f"Master Data Docs Index: {site['site_url']}")


# Get local file path to rendered Data Docs index
docs_urls = context.get_docs_sites_urls()
# Returns file path like 'file:///path/to/data_docs/local_site/index.html'
local_doc_path = docs_urls[0]["site_url"].replace("file://", "")


import os
import smtplib
import zipfile
from email.mime.application import MIMEApplication
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import great_expectations as gx

# 1. Initialize Great Expectations & Build Data Docs
#context = gx.get_context()
#context.build_data_docs()

# 2. Get local path to the Data Docs site directory
docs_urls = context.get_docs_sites_urls()
# 'file:///path/to/data_docs/local_site/index.html' -> get folder path
local_doc_index = docs_urls[0]["site_url"].replace("file://", "")
site_dir = os.path.dirname(local_doc_index)

# 3. Zip the entire Data Docs folder
zip_filename = "olist_dbt_dev_Docs_Complete.zip"

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(site_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Maintain directory structure inside the zip
            arcname = os.path.relpath(file_path, site_dir)
            zipf.write(file_path, arcname)

print(f"Zipped Data Docs from {site_dir} into {zip_filename}")

# 2. Email Setup
#export GMAIL_APP_PASSWORD = "idic ssmx itat uiav"

SENDER_EMAIL = "aalyajmalhussain@gmail.com"
APP_PASSWORD = "idic ssmx itat uiav"
#RECIPIENT_EMAIL = "ajmal_h@yahoo.com"
#RECIPIENT_EMAIL = "ajmal_h@yahoo.com,steingeez@gmail.com,hsliew2001@gmail.com,jingna83@gmail.com,Jamie.tanwc@gmail.com"
# CORRECT: Recipients must be a list of individual strings
#recipients = [
 #   "ajmal_h@yahoo.com",
 #   "steingeez@gmail.com",
 #   "hsliew2001@gmail.com",
 #   "jingna83@gmail.com",
 #   "Jamie.tanwc@gmail.com",
#]
recipients = [
    "ajmal_h@yahoo.com"
]
# If your input is a comma-separated string, split it first:
# recipients = [r.strip() for r in raw_recipient_string.split(",")]  
# In your email message headers:

msg = MIMEMultipart()
msg["From"] = SENDER_EMAIL

msg["To"] = ", ".join(recipients)
msg["Subject"] = "Great Expectations - Full Interactive Data validation Docs"

msg.attach(
    MIMEText(
        "Attached is the complete validation Data Docs suite in a ZIP file. "
        "Extract the archive and open 'index.html' to browse all validation runs and expectations.",
        "plain",
    )
)

# 5. Attach ZIP File
with open(zip_filename, "rb") as f:
    attachment = MIMEApplication(f.read(), _subtype="zip")
    attachment.add_header(
        "Content-Disposition", "attachment", filename=zip_filename
    )
    msg.attach(attachment)

# 6. Send Email via Gmail SMTP
try:
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(SENDER_EMAIL, APP_PASSWORD)
        server.sendmail(SENDER_EMAIL, recipients, msg.as_string())
    print("Full Data Docs zip sent successfully!")
except Exception as e:
    print(f"Failed to send email: {e}")

Calculating Metrics:  97%|█████████▋| 56/58 [00:50<00:01,  1.12it/s]


Master Data Docs Index: file:///tmp/tmpdo39te6x/index.html
Zipped Data Docs from /tmp/tmpdo39te6x into olist_dbt_dev_Docs_Complete.zip
Full Data Docs zip sent successfully!
